# Pandas in Practice — Olist E-Commerce Analysis

# **Mentor:** **Doris Mugah**

**Date:** Wednesday, 9 September 2026  
**Session:** Technical / Hands-On  
**Dataset:** Brazilian E-Commerce Public Dataset by Olist

## Learning Objectives

By the end of this practical session, you should be able to:

1. Load a dataset using Pandas.
2. Inspect a DataFrame.
3. Select columns and rows.
4. Filter records using conditions.
5. Sort data.
6. Use `groupby()` and aggregation.
7. Perform basic reshaping with pivot tables.
8. Identify common beginner mistakes.
9. Conduct a simple exploratory analysis and communicate findings.

> **Key idea:** We are not learning Pandas syntax for its own sake. We are using Pandas to answer real questions from data.


## 1. Import Libraries

We will mainly use **Pandas** for data manipulation and **NumPy** where useful.


In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


## 2. Load the Olist Dataset

Place the Olist CSV files in the same folder as this notebook.

The main tables we will use are:

- `olist_orders_dataset.csv`
- `olist_order_items_dataset.csv`
- `olist_products_dataset.csv`
- `olist_customers_dataset.csv`
- `olist_sellers_dataset.csv`
- `olist_order_payments_dataset.csv`
- `olist_order_reviews_dataset.csv`


## Download Olist Dataset from Kaggle

To download the dataset from Kaggle, you will need a Kaggle API token. Follow these steps:

1.  Go to [Kaggle](https://www.kaggle.com/).
2.  Log in to your account.
3.  Go to your account settings (`https://www.kaggle.com/<username>/account`).
4.  Under the 'API' section, click 'Create New API Token'. This will download a `kaggle.json` file.
5.  Upload this `kaggle.json` file to your Colab Secrets (click the key icon on the left panel), naming the secret `KAGGLE_USERNAME` for your username and `KAGGLE_KEY` for your API key. Alternatively, you can also manually input these as `KAGGLE_USERNAME` and `KAGGLE_KEY` secrets.

In [4]:
import os
from google.colab import userdata

# Install Kaggle API client
!pip install kaggle

# Set up Kaggle authentication using Colab secrets
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

# Define the dataset path on Kaggle
KAGGLE_DATASET_PATH = 'olistbr/brazilian-ecommerce'

# Create a directory for the dataset if it doesn't exist
DATA_DIR = './olist_dataset'
os.makedirs(DATA_DIR, exist_ok=True)

# Download the dataset
!kaggle datasets download -d {KAGGLE_DATASET_PATH} -p {DATA_DIR} --unzip

print(f"Dataset downloaded and extracted to {DATA_DIR}/")

Dataset URL: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
License(s): CC-BY-NC-SA-4.0
100% 42.6M/42.6M [00:01<00:00, 34.5MB/s]

Dataset downloaded and extracted to ./olist_dataset/


The Olist dataset has been downloaded. Now, let's load the main tables into Pandas DataFrames from the local files.

In [5]:
# Load the main Olist tables into Pandas DataFrames

import os

orders = pd.read_csv(os.path.join(DATA_DIR, "olist_orders_dataset.csv"))
order_items = pd.read_csv(os.path.join(DATA_DIR, "olist_order_items_dataset.csv"))
products = pd.read_csv(os.path.join(DATA_DIR, "olist_products_dataset.csv"))
customers = pd.read_csv(os.path.join(DATA_DIR, "olist_customers_dataset.csv"))
sellers = pd.read_csv(os.path.join(DATA_DIR, "olist_sellers_dataset.csv"))
order_payments = pd.read_csv(os.path.join(DATA_DIR, "olist_order_payments_dataset.csv"))
order_reviews = pd.read_csv(os.path.join(DATA_DIR, "olist_order_reviews_dataset.csv"))

print("Orders:", orders.shape)
print("Order items:", order_items.shape)
print("Products:", products.shape)
print("Customers:", customers.shape)
print("Sellers:", sellers.shape)
print("Payments:", order_payments.shape)
print("Reviews:", order_reviews.shape)


Orders: (99441, 8)
Order items: (112650, 7)
Products: (32951, 9)
Customers: (99441, 5)
Sellers: (3095, 4)
Payments: (103886, 5)
Reviews: (99224, 7)


## 3. First Look at the Data

### `head()`

Displays the first few records.


In [6]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


### `tail()`

Displays the last few records.


In [7]:
orders.tail()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28 00:00:00
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00
99440,66dea50a8b16d9b4dee7af250b4be1a5,edb027a75a1449115f6b43211ae02a24,delivered,2018-03-08 20:57:30,2018-03-09 11:20:28,2018-03-09 22:11:59,2018-03-16 13:08:30,2018-04-03 00:00:00


### `shape`

Returns the number of rows and columns.


In [8]:
orders.shape

(99441, 8)

### Column names

In [9]:
orders.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')

### `info()`

Use this to inspect column names, data types and non-null values.


In [10]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


### `describe()`

Provides descriptive statistics for numerical columns.


In [11]:
order_items.describe()

,order_item_id,price,freight_value
count,112650.000000,112650.000000,112650.000000
mean,1.197834,120.653739,19.990320
std,0.705124,183.633928,15.806405
min,1.000000,0.850000,0.000000
25%,1.000000,39.900000,13.080000
50%,1.000000,74.990000,16.260000
75%,1.000000,134.900000,21.150000
max,21.000000,6735.000000,409.680000


## 4. Selecting Columns

A single column:

```python
orders["order_status"]
```

Multiple columns:

```python
orders[["order_id", "customer_id", "order_status"]]
```


In [12]:
# Select one column
orders["order_status"].head()

,order_status
0,delivered
1,delivered
2,delivered
3,delivered
4,delivered


In [13]:
# Select multiple columns
orders[["order_id", "customer_id", "order_status"]].head()

,order_id,customer_id,order_status
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered


## 5. Selecting Rows with `.loc`

`.loc` allows us to select rows and columns by labels/conditions.


In [14]:
orders.loc[0:5, ["order_id", "customer_id", "order_status"]]

,order_id,customer_id,order_status
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered


## 6. Filtering Data

### Example: delivered orders

Filtering allows us to keep only records that meet a condition.


In [15]:
delivered_orders = orders[orders["order_status"] == "delivered"]

delivered_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [16]:
# Number of delivered orders
len(delivered_orders)

96478

### Filtering with multiple conditions

Use:

- `&` for AND
- `|` for OR

Remember to put each condition in parentheses.


In [17]:
# Delivered orders with an approval timestamp
delivered_approved = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_approved_at"].notna())
]

delivered_approved.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


## 7. Sorting Data

Sort from lowest to highest:


In [18]:
order_items.sort_values("price").head(10)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
27652,3ee6513ae7ea23bdfab5b9ab60bffcb5,1,8a3254bee785a526d548a81a9bc3c9be,96804ea39d96eb908e7c3afdb671bb9e,2018-05-04 03:55:26,0.85,18.23
48625,6e864b3f0ec71031117ad4cf46b7f2a1,1,8a3254bee785a526d548a81a9bc3c9be,96804ea39d96eb908e7c3afdb671bb9e,2018-05-02 20:30:34,0.85,18.23
87081,c5bdd8ef3c0ec420232e668302179113,2,8a3254bee785a526d548a81a9bc3c9be,96804ea39d96eb908e7c3afdb671bb9e,2018-05-07 02:55:22,0.85,22.30
57302,8272b63d03f5f79c56e9e4120aec44ef,6,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.20,7.89
57305,8272b63d03f5f79c56e9e4120aec44ef,9,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.20,7.89
57297,8272b63d03f5f79c56e9e4120aec44ef,1,270516a3f41dc035aa87d220228f844c,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.20,7.89
57306,8272b63d03f5f79c56e9e4120aec44ef,10,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.20,7.89
57304,8272b63d03f5f79c56e9e4120aec44ef,8,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.20,7.89
57298,8272b63d03f5f79c56e9e4120aec44ef,2,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.20,7.89
57301,8272b63d03f5f79c56e9e4120aec44ef,5,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.20,7.89


Sort from highest to lowest:


In [19]:
order_items.sort_values("price", ascending=False).head(10)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
3556,0812eb902a67711a1cb742b3cdaa65ae,1,489ae2aa008f021502940f251d4cce7f,e3b4998c7a498169dc7bce44e6bb6277,2017-02-16 20:37:36,6735.00,194.31
112233,fefacc66af859508bf1a7934eab1e97f,1,69c590f7ffc7bf8db97190b6cb6ed62e,80ceebb4ee9b31afb6c6a916a574a1e2,2018-08-02 04:05:13,6729.00,193.21
107841,f5136e38d1a14a4dbd87dff67da82701,1,1bdf5e6731585cf01aa8169c7028d6ad,ee27a8f15b1dded4d213a468ba4eb391,2017-06-15 02:45:17,6499.00,227.66
74336,a96610ab360d42a2e5335a3998b4718a,1,a6492cc69376c469ab6f61d8f44de961,59417c56835dd8e2e72f91f809cd4092,2017-04-18 13:25:18,4799.00,151.34
11249,199af31afc78c699f0dbf71fb178d4d4,1,c3ed642d592594bb648ff4a04cee2747,59417c56835dd8e2e72f91f809cd4092,2017-05-09 15:50:15,4690.00,74.34
62086,8dbc85d1447242f3b127dda390d56e19,1,259037a6a41845e455183f89c5035f18,c72de06d72748d1a0dfb2125be43ba63,2018-06-28 12:36:36,4590.00,91.78
29193,426a9742b533fc6fed17d1fd6d143d7e,1,a1beef8f3992dbd4cd8726796aa69c53,512d298ac2a96d1931b6bd30aa21f61d,2018-08-16 14:24:28,4399.87,113.45
45843,68101694e5c5dc7330c91e1bbc36214f,1,6cdf8fc1d741c76586d8b6b15e9eef30,ed4acab38528488b65a9a9c603ff024a,2018-04-05 08:27:27,4099.99,75.27
78310,b239ca7cd485940b31882363b52e6674,1,dd113cb02b2af9c8e5787e8f1f0722f6,821fb029fc6e495ca4f08a35d51e53a5,2018-08-02 08:15:14,4059.00,104.51
59137,86c4eab1571921a6a6e248ed312f5a5a,1,6902c1962dd19d540807d0ab8fade5c6,fa1c13f2614d7b5c4749cbc52fecda94,2017-03-23 20:08:04,3999.90,17.01


## 8. Counting Categories with `value_counts()`

This is a quick way to count how frequently each value appears.


In [20]:
orders["order_status"].value_counts()

,count
order_status,
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


## 9. `groupby()` — Group Similar Records

Think of `groupby()` as:

> **Group similar records together, then calculate something useful about each group.**

Example: number of orders for each status.


In [21]:
orders.groupby("order_status").size()

,0
order_status,
approved,2
canceled,625
created,5
delivered,96478
invoiced,314
processing,301
shipped,1107
unavailable,609


In [22]:
# Sort the status counts from highest to lowest
status_summary = (
    orders.groupby("order_status")
    .size()
    .sort_values(ascending=False)
)

status_summary

,0
order_status,
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


## 10. Aggregation

Common aggregation functions:

| Function | Meaning |
|---|---|
| `count` | Number of records |
| `sum` | Total |
| `mean` | Average |
| `min` | Minimum |
| `max` | Maximum |


In [23]:
# Basic seller-level aggregation
seller_sales = (
    order_items.groupby("seller_id")["price"]
    .sum()
    .sort_values(ascending=False)
)

seller_sales.head(10)

,price
seller_id,
4869f7a5dfa277a7dca6462dcf3b52b2,229472.63
53243585a1d6dc2643021fd1853d8905,222776.05
4a3ca9315b744ce9f8e9374361493884,200472.92
fa1c13f2614d7b5c4749cbc52fecda94,194042.03
7c67e1448b00f6e969d365cea6b010ab,187923.89
7e93a43ef30c4f03f38b393420bc753a,176431.87
da8622b14eb17ae2831f4ac5b9dab84a,160236.57
7a67c85e85bb2ce8582c35f2203ad736,141745.53
1025f0e2d44d7041d6cf58b6550e0bfa,138968.55


### Multiple aggregations

We can calculate several statistics at the same time.


In [24]:
seller_summary = (
    order_items.groupby("seller_id")
    .agg(
        total_sales=("price", "sum"),
        average_price=("price", "mean"),
        number_of_items=("order_item_id", "count")
    )
    .sort_values("total_sales", ascending=False)
)

seller_summary.head(10)

,total_sales,average_price,number_of_items
seller_id,,,
4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,198.505735,1156
53243585a1d6dc2643021fd1853d8905,222776.05,543.356220,410
4a3ca9315b744ce9f8e9374361493884,200472.92,100.892260,1987
fa1c13f2614d7b5c4749cbc52fecda94,194042.03,331.129744,586
7c67e1448b00f6e969d365cea6b010ab,187923.89,137.774113,1364
7e93a43ef30c4f03f38b393420bc753a,176431.87,518.917265,340
da8622b14eb17ae2831f4ac5b9dab84a,160236.57,103.311779,1551
7a67c85e85bb2ce8582c35f2203ad736,141745.53,121.046567,1171
1025f0e2d44d7041d6cf58b6550e0bfa,138968.55,97.316912,1428


## 11. Basic Reshaping with a Pivot Table

A pivot table reorganizes data to make summaries easier to inspect.


In [25]:
seller_pivot = pd.pivot_table(
    order_items,
    values="price",
    index="seller_id",
    aggfunc="sum"
)

seller_pivot.sort_values("price", ascending=False).head(10)

,price
seller_id,
4869f7a5dfa277a7dca6462dcf3b52b2,229472.63
53243585a1d6dc2643021fd1853d8905,222776.05
4a3ca9315b744ce9f8e9374361493884,200472.92
fa1c13f2614d7b5c4749cbc52fecda94,194042.03
7c67e1448b00f6e969d365cea6b010ab,187923.89
7e93a43ef30c4f03f38b393420bc753a,176431.87
da8622b14eb17ae2831f4ac5b9dab84a,160236.57
7a67c85e85bb2ce8582c35f2203ad736,141745.53
1025f0e2d44d7041d6cf58b6550e0bfa,138968.55


## 12. Missing Values

Check missing values in every column:


In [26]:
orders.isnull().sum()

,0
order_id,0
customer_id,0
order_status,0
order_purchase_timestamp,0
order_approved_at,160
order_delivered_carrier_date,1783
order_delivered_customer_date,2965
order_estimated_delivery_date,0


Check whether a specific column contains missing values:

In [27]:
orders["order_approved_at"].isnull().sum()

np.int64(160)

Filter records where a value is missing:

In [28]:
orders[orders["order_approved_at"].isnull()].head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
1130,00b1cb0320190ca0daa2c88b35206009,3532ba38a3fd242259a514ac2b6ae6b6,canceled,2018-08-28 15:26:39,NaN,NaN,NaN,2018-09-12 00:00:00
1801,ed3efbd3a87bea76c2812c66a0b32219,191984a8ba4cbb2145acb4fe35b69664,canceled,2018-09-20 13:54:16,NaN,NaN,NaN,2018-10-17 00:00:00
1868,df8282afe61008dc26c6c31011474d02,aa797b187b5466bc6925aaaa4bb3bed1,canceled,2017-03-04 12:14:30,NaN,NaN,NaN,2017-04-10 00:00:00
2029,8d4c637f1accf7a88a4555f02741e606,b1dd715db389a2077f43174e7a675d07,canceled,2018-08-29 16:27:49,NaN,NaN,NaN,2018-09-13 00:00:00
2161,7a9d4c7f9b068337875b95465330f2fc,7f71ae48074c0cfec9195f88fcbfac55,canceled,2017-05-01 16:12:39,NaN,NaN,NaN,2017-05-30 00:00:00


## 13. Working with Dates

Convert a timestamp column from text to a datetime type.


In [29]:
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

orders[["order_purchase_timestamp"]].head()

,order_purchase_timestamp
0,2017-10-02 10:56:33
1,2018-07-24 20:41:37
2,2018-08-08 08:38:49
3,2017-11-18 19:28:06
4,2018-02-13 21:18:39


In [30]:
# Extract year and month
orders["purchase_year"] = orders["order_purchase_timestamp"].dt.year
orders["purchase_month"] = orders["order_purchase_timestamp"].dt.month

orders[["order_purchase_timestamp", "purchase_year", "purchase_month"]].head()

,order_purchase_timestamp,purchase_year,purchase_month
0,2017-10-02 10:56:33,2017,10
1,2018-07-24 20:41:37,2018,7
2,2018-08-08 08:38:49,2018,8
3,2017-11-18 19:28:06,2017,11
4,2018-02-13 21:18:39,2018,2


# 14. Common Beginner Mistakes

### Mistake 1: Missing parentheses with multiple conditions

**Incorrect:**
```python
df[df["price"] > 100 & df["price"] < 500]
```

**Correct:**
```python
df[(df["price"] > 100) & (df["price"] < 500)]
```

### Mistake 2: Using `and` instead of `&`

Use `&` for element-wise AND and `|` for element-wise OR.

### Mistake 3: Misspelling column names

Always check:

```python
df.columns
```

### Mistake 4: Forgetting `ascending=False`

```python
df.sort_values("price", ascending=False)
```

gives highest values first.

### Mistake 5: Not checking data types

Use:

```python
df.info()
```

### Mistake 6: Date columns stored as text

Use:

```python
pd.to_datetime()
```


# 15. Pandas Quick Reference

```python
import pandas as pd

pd.read_csv()
df.head()
df.tail()
df.shape
df.columns
df.info()
df.describe()

df["column"]
df[["column1", "column2"]]

df.loc[]
df[df["column"] == value]

df.sort_values()
df.value_counts()

df.groupby()
df.agg()

df.isnull()
df.notna()

pd.pivot_table()
pd.to_datetime()
```

